<b>General pipeline</b>: preprocessing => segmentation => feature extraction => classification (https://peerj.com/articles/cs-620/#fig-5)

<h2><b> IDEAS FROM ARTICLE WITH COMPARISON OF METHODS </b></h2>
Quality improvement: Contrast stretching, Grayscale stretching, Log transformation, Gamma correction,
Image negative, Histogram equalization methods, Adaptive local contrast stretching

Filtering (Gaussian, Poisson, and Quantum noise are different types of
noise artifacts - which one do we have?? because 'If we try to minimize one
class of noise, it may disrupt the other'): Average filter,
Bilateral filter, Laplacian filter, Homomorphic filter, and Butterworth filter, Median
Gaussian filter, and Weiner filter, Median (reduces boundaries), Gaussian (reduces picture information)

Segmentation: segmentation of panoramic X-rays using wavelet transformation shows
better results than adaptive and iterative thresholding, template matching technique, Otsu’s threshold combined with morphological dilation, (gap valley extraction, modified canny
edge detector, guided iterative contour tracing, and template matching), contour-based segmentation,  horizontal integral projection, computing moments and statistical characteristics, edge segmentation methods: Canny
and Sobel, Quantum Particle Swarm Optimization (QPSO)  employed for
multilevel thresholding,  Gaussian kernel-based conditional spatial
fuzzy c-means (GK-csFCM) clustering algorithm.

Classic ML: Feature extraction (Projected principal edge distribution
(PPED) + Geometric properties +
Region descriptors) + SVM, Segmentation of mandibular teeth carried
out by applying Random forest regression-
voting constrained local model (RFRV-
CLM) in two steps: The 1st step gives an
estimate of individual teeth and mandible
regions used to initialize search for the
tooth. In the second step, the investigation
is carried out separately for each tooth.


<h2><b> SUGGESTION AFTER CONSULTING WITH SOME EXPERTS IN ALL TOPICS :D </b></h2>


1. preprocessing:
- CLAHE +
- every noise reduction that do not blur edges (unsharp masking / bilateral filtering / non-local mean)
- morphological operation (erosion / dilatation etc. - check before or after first segmentation)
- validate visually + check strength of found edges (Sobel/Canny)
- Gamma correction
-  Log transformation
- Median filter, Gaussian filter, Wiener filter (check what type of noise there is primarily [SaltNPepper / Gaussian / Photon(Quantum) / ... ]
- contrast stretching
- adaptive thresholding
- averaging (maybe images without segments that have teeth in them, to find some pattern of the noise, that we can possibly remove)
- top-hat (white / black)
- FFT where high value -> set to zero
- Log Gabor [for Speckle noise]
2. segmentation:
- watershed with markers on each tooth
- active contours (snake etc)
- teeth touching / overlapping = concavity analysis + find ways to count teeth properly (opening / closure)
- Validate segmentation (compute Dice coefficient vs ground truth), IoU, visually how they look compared to manually prepared, check number of teeth
- remove very small/large segments
- Laplacian filter for edges
- Otsu's threshold + morphological dilation
- Contour-based segmentation (snake after initialization of predicted space that our teeth should be / level-set)
- Wavelet transformation
3. Feature extraction:
- hu moments, aspect ratio, solidity, circularity/compactness, eccentricity of fitted ellipse
- Position/context features: centroid position, orientation angle, number of neighbors and distances to them, relative position in dental arch
- Texture features (often overlooked but useful): Local Binary Patterns (LBP) on tooth region, Gray-level statistic
- plot how clusterization works (of types / sides / jaws)
- Projected Principal Edge Distribution + Shape Descriptors + Region descriptors (texture etc.) => SVM
4. Classification:
- two-step: tooth type classification (SVM / RF with extracted features)
- second: get specific tooth class (have type, have orientation [bottom/top jaw, left/right part of mouth])
- spatial graph of teeth, compare to template
- cross - validation
- we can even try to extract as many features to try to just generate one class (1..32) for every tooth (easier if we want to see some results)
- graph data: centroid position, tooth type, jaw (top/bottom), neighbors (teeth within certain distance)
- prepare template from all images (average / median positions by teeth type+class, distances, etc.)
- Random Forest estimates approximate regions for each tooth (search regions) => CLM (Constrained Local Model) searches for a tooth in every region
5. Missing teeth:
- check distances between neighbours
- compare to template to see where the tooth should be if dist > expected, mark as missing
-

In [ ]:
import numpy as np
import cv2
import json
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import kagglehub

In [ ]:
# Load images

def get_images(download_dataset_flag):
    if download_dataset_flag:
        dataset_path = kagglehub.dataset_download("humansintheloop/teeth-segmentation-on-dental-x-ray-images")
        sources = {k: Path(dataset_path) / f'Teeth Segmentation {k}' for k in ['JSON', 'PNG']}
    else:
        sources = {k: Path(f'./Teeth Segmentation {k}') for k in ['JSON', 'PNG']}

    meta = {}
    for p in sources.values():
        meta.update(json.loads((p / 'meta.json').read_text()))

    images = [
        (Image.open(img_path), img_path)
        for p in sources.values()
        for img_path in (p / 'd2' / 'img').glob('*')
    ]

    images_np = [(np.array(image), str(p)) for (image, p) in images[:len(images)//2]]
    return images_np

In [ ]:
DOWNLOAD_DATASET = False

images_np = get_images(DOWNLOAD_DATASET)

<h1><b>STEP1:</b> Normalize images, resize or truncate to one size </h1>

In [ ]:
# Get general info about images size

shapes = []
for image, p in images_np:
    shapes.append(image.shape)

min_width = np.min(np.array(shapes)[:, 1])
max_width = np.max(np.array(shapes)[:, 1])
min_height = np.min(np.array(shapes)[:, 0])
max_height = np.max(np.array(shapes)[:, 0])
avg_width = np.median(np.array(shapes)[:, 1])
avg_height = np.median(np.array(shapes)[:, 0])

print(min_width, max_width, min_height, max_height, avg_width, avg_height)


In [ ]:
# Method changes image size and calculates new polygon positions (but do not save them so we need to either run it every time of rewrite it to save new data

def normalize_image_size(image, image_metadata, output_height=1024, output_width=2045):
    image_height, image_width = image.shape

    scale_y = output_height / image_height
    scale_x = output_width / image_width

    resized_image = cv2.resize(image, (output_width, output_height))
    image_metadata['size'] = {
        'height': output_height,
        'width': output_width,
    }

    for segment_no in range(len(image_metadata['objects'])):
        for vertex_no in range(len(image_metadata['objects'][segment_no]['points']['exterior'])):
            x = image_metadata['objects'][segment_no]['points']['exterior'][vertex_no][0]
            y = image_metadata['objects'][segment_no]['points']['exterior'][vertex_no][1]
            image_metadata['objects'][segment_no]['points']['exterior'][vertex_no][0] = int(x*scale_x)
            image_metadata['objects'][segment_no]['points']['exterior'][vertex_no][1] = int(y*scale_y)

    return resized_image, image_metadata

In [ ]:
import copy

# Method transforms image_number raw images and metadata to resized images and metadata
def adjust_images(images_with_paths, image_number=None):
    normalized_images = []
    images_with_metadata = []
    for image, image_path in images_with_paths[:image_number if image_number else len(images_with_paths)]:
        metadata_filename = image_path.split('\\')[-1] + '.json'
        metadata_path = '\\'.join(image_path.split('\\')[:-2]) + '\\ann\\' + metadata_filename
        with open(metadata_path, 'r') as f:
            image_metadata = json.load(f)
        images_with_metadata.append((image, image_metadata, image_path))
        resized_image, resized_image_metadata = normalize_image_size(image, copy.deepcopy(image_metadata))
        normalized_images.append((resized_image, resized_image_metadata))

    return normalized_images, images_with_metadata

In [ ]:
resized_images_np, original_images_np = adjust_images(images_np)

In [ ]:
# Compare how image and shapes created from metadata look before and after resize

def display_images_size_compared():
    i = 1
    ipo = 20
    for (resized_image, resized_metadata), (original_image, metadata, path) in zip(resized_images_np[i*ipo: (i+1)*ipo], original_images_np[i*ipo: (i+1)*ipo]):
        if image.shape[1] < 2100:
            print(path)
            original_xs = []
            original_ys = []
            for segment_no in range(len(metadata['objects'])):
                arr = np.array(metadata['objects'][segment_no]['points']['exterior'])
                arr = np.concatenate((arr, np.array(metadata['objects'][segment_no]['points']['exterior'][0]).reshape(1,2)))
                original_xs.append(arr[:, 0])
                original_ys.append(arr[:, 1])

            resized_xs = []
            resized_ys = []
            for segment_no in range(len(resized_metadata['objects'])):
                arr = np.array(resized_metadata['objects'][segment_no]['points']['exterior'])
                arr = np.concatenate((arr, np.array(resized_metadata['objects'][segment_no]['points']['exterior'][0]).reshape(1,2)))
                resized_xs.append(arr[:, 0])
                resized_ys.append(arr[:, 1])

            fig, axes = plt.subplots(1, 2, figsize=(20, 10))
            axes[1].imshow(resized_image, cmap='gray')
            axes[1].set_title(f'Resized image {resized_image.shape[0]}x{resized_image.shape[1]}')
            for x, y in zip(resized_xs, resized_ys):
                axes[1].plot(x, y)
            axes[1].axis('off')

            axes[0].imshow(image, cmap='gray')
            axes[0].set_title(f'Original image {image.shape[0]}x{image.shape[1]}')
            for x, y in zip(original_xs, original_ys):
                axes[0].plot(x, y)
            axes[0].axis('off')

            plt.show()

<h1><b>STEP2:</b> Find out what type of noise is present on the images </h1>

https://scikit-image.org/docs/stable/api/skimage.restoration.html#skimage.restoration.estimate_sigma

In [ ]:
# TODO: its not crucial I think, but it would be nice to have to say / write some words about what are the biggest issues with our data quality
def get_noise_report(images):
    pass

<h1><b>STEP3:</b> Image denoising </h1>

In [ ]:
def apply_CLAHE(images, noise_info):
    improved_images = []
    for image in images:
        # try what params you think works best, I suggest tileGridSize between 8 and 16, clipLimit between 2 and 16
        fig, axes = plt.subplots(1, 2, figsize=(20, 20))
        axes[0].imshow(image, cmap='gray')
        axes[0].set_title(f'Original')
        axes[0].axis('off')

        clahe = cv2.createCLAHE(clipLimit=4, tileGridSize=(8,8))
        image_enhanced = clahe.apply(image)
        improved_images.append(image_enhanced)

        axes[1].imshow(image_enhanced, cmap='gray')
        axes[1].set_title(f'After CLAHE (clipLimit={clahe.getClipLimit()}, tileGridSize={clahe.getTilesGridSize()})')
        axes[1].axis('off')

        plt.show()
        plt.close()


        # Code for comparing CLAHE params
        # for clip_limit in [2, 4, 8, 16]:
        #     fig, axes = plt.subplots(2, 2, figsize=(20, 10))
        #     axes[0,0].imshow(image, cmap='gray')
        #     axes[0,0].set_title(f'Original')
        #     axes[0,0].axis('off')
        #     next_axes = [0,1]
        #     for tile_grid_size in [(4,4), (8,8), (16,16)]:
        #         clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
        #         image_enhanced = clahe.apply(image)
        #
        #         axes[next_axes[0], next_axes[1]].imshow(image_enhanced, cmap='gray')
        #         axes[next_axes[0], next_axes[1]].set_title(f'After CLAHE (clipLimit={clip_limit}, tileGridSize={tile_grid_size})')
        #         axes[next_axes[0], next_axes[1]].axis('off')
        #
        #         next_axes[1] = next_axes[1] + 1
        #         if next_axes[1] == 2:
        #             next_axes = [next_axes[0]+1, 0]
        #
        #     plt.show()
        #     plt.close()
    return improved_images

In [ ]:
# Images after CLAHE
improved_images_np = apply_CLAHE([img for img, _ in resized_images_np][:20], noise_report)

In [ ]:
def apply_gamma_correction(images, noise_info):
    gamma = 2
    inv_gamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** inv_gamma) * 255
                      for i in range(256)]).astype("uint8")
    improved_images = []
    for image in images:
        # maybe gamma correction? it looks promising with gamma=2 but you can try something different
        fig, axes = plt.subplots(1, 2, figsize=(20, 20))
        axes[0].imshow(image, cmap='gray')
        axes[0].set_title(f'Original')
        axes[0].axis('off')

        image_enhanced = cv2.LUT(image, table)
        improved_images.append(image_enhanced)

        axes[1].imshow(image_enhanced, cmap='gray')
        axes[1].set_title(f'After gamma correction (gamma = {gamma})')
        axes[1].axis('off')

        plt.show()
        plt.close()

        # Code for comparing gammas
        # fig, axes = plt.subplots(2, 2, figsize=(20, 10))
        # axes[0,0].imshow(image, cmap='gray')
        # axes[0,0].set_title(f'Original')
        # axes[0,0].axis('off')
        # next_axes = [0,1]
        # for gamma in [0.5, 2, 3]:
        #     inv_gamma = 1.0 / gamma
        #     table = np.array([((i / 255.0) ** inv_gamma) * 255
        #                       for i in range(256)]).astype("uint8")
        #
        #     image_enhanced = cv2.LUT(image, table)
        #
        #     axes[next_axes[0], next_axes[1]].imshow(image_enhanced, cmap='gray')
        #     axes[next_axes[0], next_axes[1]].set_title(f'After gamma correction (gamma={gamma})')
        #     axes[next_axes[0], next_axes[1]].axis('off')
        #
        #     next_axes[1] = next_axes[1] + 1
        #     if next_axes[1] == 2:
        #         next_axes = [next_axes[0]+1, 0]
        #
        # plt.show()
        # plt.close()
    return improved_images

In [ ]:
# Images after gamma correction
improved_images_np2 = apply_gamma_correction([img for img, _ in resized_images_np][:20], noise_report)

In [ ]:
# Gamma correction after CLAHE (too much I think, but you can try

improved_images_np3 = apply_gamma_correction(improved_images_np[:20], noise_report)

In [ ]:
# CLAHE after gamma correction (dont know whether it is different then CLAHE only but you can try)

improved_images_np4 = apply_CLAHE(improved_images_np2[:20], noise_report)

In [ ]:
def apply_bilateral_filter(images, noise_info):
    d = 4
    sigma_color = 50
    sigma_space = 50

    improved_images = []
    for image in images:
        fig, axes = plt.subplots(1, 2, figsize=(20, 20))
        axes[0].imshow(image, cmap='gray')
        axes[0].set_title(f'Original')
        axes[0].axis('off')

        image_filtered = cv2.bilateralFilter(image, d=d, sigmaColor=sigma_color, sigmaSpace=sigma_space)

        axes[1].imshow(image_filtered, cmap='gray')
        axes[1].set_title(f'After bilateral filter (d={d}, sigmaColor={sigma_color}, sigmaSpace={sigma_space})')
        axes[1].axis('off')

        plt.show()
        plt.close()

    return improved_images

In [ ]:
# Bilateral filter after CLAHE
filtered_images = apply_bilateral_filter(improved_images_np, noise_report)

<h1><b>STEP4:</b> Segmentation </h1>

<h1><b>STEP5:</b> Add after segmentation processing <i>(optional)</i> </h1>

<h1><b>STEP6:</b> Extract features from segments </h1>

<h1><b>STEP7:</b> Model for tooth type classification </h1>

<h1><b>STEP8:</b> Prepare tooth data for template </h1>

<h1><b>STEP9:</b> Create template map </h1>

<h1><b>STEP10:</b> Generate summary of teeth status (whole pipeline + model for graph comparaison with template) </h1>